In [1]:
# Check and install required packages
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Install advanced model libraries
try:
    import timm
except ImportError:
    install_package("timm")
    
try:
    import segmentation_models_pytorch
except ImportError:
    install_package("segmentation-models-pytorch")

print("All required packages installed successfully!")

import os
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import precision_recall_fscore_support, jaccard_score
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import timm
import segmentation_models_pytorch as smp

# Set the seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 72.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [1]:
# -*- coding: utf-8 -*-
"""
Advanced Oil Spill Segmentation Pipeline for SAR Imagery (Multi-GPU Enabled)

This script loads a pre-trained SegFormer model and continues training it.
"""

import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, jaccard_score
from tqdm.notebook import tqdm
from transformers import SegformerForSemanticSegmentation, SegformerConfig
from torch.amp import autocast, GradScaler
import torch.nn.functional as F
from collections import OrderedDict

# =====================================================================================
# 0. Configuration Block
# =====================================================================================
class Config:
    TRAIN_IMG_DIR = '/kaggle/input/sentinel-dataset/sentinel_dataset/train/image'
    TRAIN_LBL_DIR = '/kaggle/input/sentinel-dataset/sentinel_dataset/train/label'
    VAL_IMG_DIR   = '/kaggle/input/sentinel-dataset/sentinel_dataset/test/image'
    VAL_LBL_DIR   = '/kaggle/input/sentinel-dataset/sentinel_dataset/test/label'
    
    # --- CHECKPOINT TO LOAD ---
    PRETRAINED_CHECKPOINT = '/kaggle/input/segformer/pytorch/default/1/best_model_continued (2).pth'
    
    NUM_GPUS = torch.cuda.device_count()

    MODEL_NAME = 'nvidia/segformer-b4-finetuned-ade-512-512'
    IMAGE_SIZE = (256, 256)
    BASE_BATCH_SIZE = 8
    BATCH_SIZE = BASE_BATCH_SIZE * NUM_GPUS if NUM_GPUS > 0 else BASE_BATCH_SIZE
    
    # --- TRAINING DURATION FOR THIS RUN ---
    EPOCHS = 100 # Continue training for 100 more epochs
    
    BASE_LR = 1e-4
    WEIGHT_DECAY = 1e-5
    SEED = 42
    
    T_0 = 10
    T_MULT = 1
    ETA_MIN = 1e-6

    APPLY_POST_PROCESSING = True
    MORPH_KERNEL_SIZE = (3, 3)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if Config.NUM_GPUS > 0:
    print(f"Found {Config.NUM_GPUS} GPUs. Effective batch size: {Config.BATCH_SIZE}")


# For reproducibility
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(Config.SEED)

# =====================================================================================
# 1. Advanced SAR Image Preprocessing (Speckle Filtering)
# =====================================================================================
def lee_filter(img, win_size):
    img_mean = cv2.blur(img, (win_size, win_size))
    img_sqr_mean = cv2.blur(img**2, (win_size, win_size))
    img_var = img_sqr_mean - img_mean**2
    overall_var = np.var(img)
    weights = img_var / (img_var + overall_var + 1e-8)
    if len(weights.shape) < len(img.shape):
        weights = np.expand_dims(weights, axis=-1)
    filtered_img = img_mean + weights * (img - img_mean)
    return filtered_img.astype(np.uint8)

def enhanced_sar_preprocessing(image_path, target_size=(256, 256)):
    sar_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if sar_image is None:
        raise FileNotFoundError(f"Image not found at {image_path}")
    img_float = sar_image.astype(np.float32)
    filtered_img = lee_filter(img_float, win_size=7)
    norm_img = cv2.normalize(filtered_img, None, 0, 255, cv2.NORM_MINMAX)
    img_uint8 = norm_img.astype(np.uint8)
    img_resized = cv2.resize(img_uint8, target_size, interpolation=cv2.INTER_AREA)
    return img_resized

# =====================================================================================
# 2. Data Augmentation
# =====================================================================================
train_aug = A.Compose([
    A.Resize(height=Config.IMAGE_SIZE[0], width=Config.IMAGE_SIZE[1]),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ElasticTransform(p=0.2, alpha=120, sigma=120 * 0.05),
    A.GridDistortion(p=0.2),
    A.OpticalDistortion(distort_limit=0.2, p=0.2),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])

val_aug = A.Compose([
    A.Resize(height=Config.IMAGE_SIZE[0], width=Config.IMAGE_SIZE[1]),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])

# =====================================================================================
# 3. PyTorch Dataset Class
# =====================================================================================
class OilSpillDataset(Dataset):
    def __init__(self, image_dir, label_dir, preprocess_fn, augmentations=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.preprocess_fn = preprocess_fn
        self.augmentations = augmentations
        self.file_list = sorted([f for f in os.listdir(image_dir) if f in os.listdir(label_dir)])

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        fname = self.file_list[idx]
        img_path = os.path.join(self.image_dir, fname)
        lbl_path = os.path.join(self.label_dir, fname)

        img = self.preprocess_fn(img_path, target_size=Config.IMAGE_SIZE)
        lbl = cv2.imread(lbl_path, cv2.IMREAD_GRAYSCALE)
        if lbl is None:
            raise FileNotFoundError(f"Label not found at {lbl_path}")
        lbl = (lbl > 0).astype(np.uint8)

        img = np.expand_dims(img, axis=-1)

        if self.augmentations:
            augmented = self.augmentations(image=img, mask=lbl)
            img, lbl = augmented['image'], augmented['mask']
        
        return img, lbl.unsqueeze(0).float()

# =====================================================================================
# 4. Advanced Loss Functions (Dice & Lovász-Hinge)
# =====================================================================================
def lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1. - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard

class LovaszHingeLoss(nn.Module):
    def __init__(self, per_image=True):
        super().__init__()
        self.per_image = per_image

    def forward(self, logits, labels):
        if self.per_image:
            loss = torch.mean(torch.stack([self.lovasz_hinge_flat(logit.flatten(), label.flatten()) 
                                           for logit, label in zip(logits, labels)]))
        else:
            loss = self.lovasz_hinge_flat(logits.flatten(), labels.flatten())
        return loss

    @staticmethod
    def lovasz_hinge_flat(logits, labels):
        if len(labels) == 0:
            return logits.sum() * 0.
        signs = 2. * labels.float() - 1.
        errors = (1. - logits * signs)
        errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
        perm = perm.data
        gt_sorted = labels[perm]
        grad = lovasz_grad(gt_sorted)
        loss = torch.dot(F.relu(errors_sorted), grad)
        return loss

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        dice = (2. * intersection + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)
        return 1 - dice

class HybridLovaszDiceLoss(nn.Module):
    def __init__(self, lovasz_weight=1.0, dice_weight=1.0):
        super().__init__()
        self.lovasz = LovaszHingeLoss(per_image=True)
        self.dice = DiceLoss()
        self.lovasz_weight = lovasz_weight
        self.dice_weight = dice_weight

    def forward(self, inputs, targets):
        lovasz_loss = self.lovasz(inputs, targets)
        dice_loss = self.dice(inputs, targets)
        return self.lovasz_weight * lovasz_loss + self.dice_weight * dice_loss

# =====================================================================================
# 5. Model Definition (SegFormer)
# =====================================================================================
def build_model():
    """Builds and configures the SegFormer model, adapting it for 1-channel input."""
    
    config = SegformerConfig.from_pretrained(Config.MODEL_NAME)
    
    config.num_channels = 1
    config.num_labels = 1
    
    model = SegformerForSemanticSegmentation.from_pretrained(
        Config.MODEL_NAME,
        config=config,
        ignore_mismatched_sizes=True,
    )

    original_weights = SegformerForSemanticSegmentation.from_pretrained(Config.MODEL_NAME).segformer.encoder.patch_embeddings[0].proj.weight.data
    new_weights = original_weights.mean(dim=1, keepdim=True)
    model.segformer.encoder.patch_embeddings[0].proj.weight.data = new_weights
    
    return model.to(device)

# =====================================================================================
# 6. Post-Processing for Mask Refinement
# =====================================================================================
def refine_mask(mask_np):
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, Config.MORPH_KERNEL_SIZE)
    opened_mask = cv2.morphologyEx(mask_np, cv2.MORPH_OPEN, kernel, iterations=1)
    closed_mask = cv2.morphologyEx(opened_mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    return closed_mask

# =====================================================================================
# 7. Training and Validation Loops
# =====================================================================================
def train_epoch(model, loader, criterion, optimizer, scaler, scheduler):
    model.train()
    running_loss = 0.0
    loop = tqdm(loader, desc='Training', leave=False)
    for imgs, masks in loop:
        imgs, masks = imgs.to(device), masks.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        
        with autocast(device_type="cuda"):
            outputs = model(pixel_values=imgs).logits
            preds = F.interpolate(outputs, size=masks.shape[2:], mode='bilinear', align_corners=False)
            loss = criterion(preds, masks)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        running_loss += loss.item()
        loop.set_postfix(loss=loss.item(), lr=optimizer.param_groups[0]['lr'])
        
    return running_loss / len(loader)

def validate(model, loader, criterion):
    model.eval()
    val_loss = 0.0
    all_preds, all_masks = [], []
    
    loop = tqdm(loader, desc='Validating', leave=False)
    with torch.no_grad():
        for imgs, masks in loop:
            imgs, masks = imgs.to(device), masks.to(device)
            
            with autocast(device_type="cuda"):
                outputs = model(pixel_values=imgs).logits
                preds = F.interpolate(outputs, size=masks.shape[2:], mode='bilinear', align_corners=False)
                loss = criterion(preds, masks)

            val_loss += loss.item()
            bin_preds = (torch.sigmoid(preds) > 0.5).cpu().numpy().astype(np.uint8)
            
            if Config.APPLY_POST_PROCESSING:
                refined_preds = np.array([refine_mask(p.squeeze(0)) for p in bin_preds]).astype(np.uint8)
                all_preds.append(refined_preds.flatten())
            else:
                all_preds.append(bin_preds.flatten())
                
            all_masks.append(masks.cpu().numpy().flatten())
            loop.set_postfix(loss=loss.item())

    preds_arr = np.concatenate(all_preds)
    masks_arr = np.concatenate(all_masks)
    
    if masks_arr.size == 0:
        return val_loss / len(loader), 0.0, 0.0, 0.0, 0.0
        
    p, r, f, _ = precision_recall_fscore_support(masks_arr, preds_arr, average='binary', zero_division=0)
    iou = jaccard_score(masks_arr, preds_arr, average='binary', zero_division=0)
    
    return val_loss / len(loader), p, r, f, iou

# =====================================================================================
# 8. Main Execution Block
# =====================================================================================
def main():
    # --- Data Setup ---
    train_dataset = OilSpillDataset(
        image_dir=Config.TRAIN_IMG_DIR,
        label_dir=Config.TRAIN_LBL_DIR,
        preprocess_fn=enhanced_sar_preprocessing,
        augmentations=train_aug
    )
    val_dataset = OilSpillDataset(
        image_dir=Config.VAL_IMG_DIR,
        label_dir=Config.VAL_LBL_DIR,
        preprocess_fn=enhanced_sar_preprocessing,
        augmentations=val_aug
    )
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    print(f"Found {len(train_dataset)} training images and {len(val_dataset)} validation images.")

    # --- Model Loading ---
    model = build_model()
    try:
        # Clean the state dict keys if they start with 'module.' from DataParallel
        state_dict = torch.load(Config.PRETRAINED_CHECKPOINT)
        new_state_dict = OrderedDict()
        for k, v in state_dict.items():
            name = k[7:] if k.startswith('module.') else k
            new_state_dict[name] = v
        model.load_state_dict(new_state_dict)
        print(f"✅ Model weights loaded successfully from {Config.PRETRAINED_CHECKPOINT}")
    except FileNotFoundError:
        print(f"Info: No checkpoint found at {Config.PRETRAINED_CHECKPOINT}. Starting training from scratch.")
    except Exception as e:
        print(f"Error loading model weights: {e}. Starting training from scratch.")

    if Config.NUM_GPUS > 1:
        print(f"Using nn.DataParallel for {Config.NUM_GPUS} GPUs.")
        model = nn.DataParallel(model)

    # --- Establish Baseline Performance ---
    print("\nRunning initial validation to establish baseline IoU...")
    criterion = HybridLovaszDiceLoss()
    val_loss, p, r, f, baseline_iou = validate(model, val_loader, criterion)
    best_val_iou = baseline_iou
    print(f"Baseline IoU of the loaded model: {best_val_iou:.4f}")

    # --- Training Setup ---
    optimizer = optim.AdamW(model.parameters(), lr=Config.BASE_LR, weight_decay=Config.WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=Config.T_0, T_mult=Config.T_MULT, eta_min=Config.ETA_MIN
    )
    scaler = GradScaler()
    history = []

    # --- Training Loop ---
    for epoch in range(Config.EPOCHS):
        print(f"\n{'='*20} EPOCH {epoch+1}/{Config.EPOCHS} {'='*20}")
        
        tr_loss = train_epoch(model, train_loader, criterion, optimizer, scaler, scheduler)
        val_loss, p, r, f, iou = validate(model, val_loader, criterion)
        
        print(f"Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f}")
        print(f"Precision: {p:.4f} | Recall: {r:.4f} | F1-Score: {f:.4f} | IoU: {iou:.4f}")
        
        history.append({
            'epoch': epoch + 1, 'train_loss': tr_loss, 'val_loss': val_loss,
            'precision': p, 'recall': r, 'f1_score': f, 'iou': iou
        })
        
        if iou > best_val_iou:
            best_val_iou = iou
            state_dict_to_save = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(state_dict_to_save, 'best_model_continued.pth')
            print(f"✅ Saved new best model with IoU: {best_val_iou:.4f}")

    print(f"\n{'='*20} CONTINUED TRAINING SUMMARY {'='*20}")
    history_df = pd.DataFrame(history)
    print(history_df.to_string())
    print(f"\nBest Validation IoU achieved in this run: {best_val_iou:.4f}")

if __name__ == '__main__':
    main()

2025-07-24 06:46:29.189550: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753339589.211709     131 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753339589.218475     131 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Using device: cuda
Found 2 GPUs. Effective batch size: 16
Found 3354 training images and 839 validation images.


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b4-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- segformer.encoder.patch_embeddings.0.proj.weight: found shape torch.Size([64, 3, 7, 7]) in the checkpoint and torch.Size([64, 1, 7, 7]) in the model instantiated
- decode_head.classifier.weight: found shape torch.Size([150, 768, 1, 1]) in the checkpoint and torch.Size([1, 768, 1, 1]) in the model instantiated
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model weights loaded successfully from /kaggle/input/segformer/pytorch/default/1/best_model_continued (2).pth
Using nn.DataParallel for 2 GPUs.

Running initial validation to establish baseline IoU...


Validating:   0%|          | 0/53 [00:00<?, ?it/s]

...

Training:   0%|          | 0/210 [00:00<?, ?it/s]

Validating:   0%|          | 0/53 [00:00<?, ?it/s]

Train Loss: 0.2685 | Val Loss: 1.2047
Precision: 0.9040 | Recall: 0.9248 | F1-Score: 0.9143 | IoU: 0.8421

==================== EPOCH 64/100 ====================


Training:   0%|          | 0/210 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [2]:
# -*- coding: utf-8 -*-
"""
Final Evaluation Script for SAR Oil Spill Segmentation Model

This script loads a final, trained model and evaluates its performance
on the validation dataset, calculating all relevant metrics.
"""

import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, jaccard_score
from tqdm.notebook import tqdm
from transformers import SegformerForSemanticSegmentation, SegformerConfig
from torch.amp import autocast
import torch.nn.functional as F
from collections import OrderedDict

# =====================================================================================
# 0. Configuration Block
# =====================================================================================
class Config:
    # --- PATHS ---
    VAL_IMG_DIR   = '/kaggle/input/sentinel-dataset/sentinel_dataset/test/image'
    VAL_LBL_DIR   = '/kaggle/input/sentinel-dataset/sentinel_dataset/test/label'
    
    # --- IMPORTANT: SET THE PATH TO YOUR FINAL MODEL ---
    FINAL_MODEL_PATH = '/kaggle/working/best_model_continued.pth'
    
    # --- MODEL & DATA CONFIGURATION ---
    MODEL_NAME = 'nvidia/segformer-b4-finetuned-ade-512-512'
    IMAGE_SIZE = (256, 256)
    BATCH_SIZE = 16 * torch.cuda.device_count() if torch.cuda.is_available() else 16
    
    # --- POST-PROCESSING ---
    APPLY_POST_PROCESSING = True
    MORPH_KERNEL_SIZE = (3, 3)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# =====================================================================================
# 1. Preprocessing and Augmentations
# =====================================================================================
def lee_filter(img, win_size):
    img_mean = cv2.blur(img, (win_size, win_size))
    img_sqr_mean = cv2.blur(img**2, (win_size, win_size))
    img_var = img_sqr_mean - img_mean**2
    overall_var = np.var(img)
    weights = img_var / (img_var + overall_var + 1e-8)
    if len(weights.shape) < len(img.shape):
        weights = np.expand_dims(weights, axis=-1)
    filtered_img = img_mean + weights * (img - img_mean)
    return filtered_img.astype(np.uint8)

def enhanced_sar_preprocessing(image_path, target_size=(256, 256)):
    sar_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if sar_image is None:
        raise FileNotFoundError(f"Image not found at {image_path}")
    img_float = sar_image.astype(np.float32)
    filtered_img = lee_filter(img_float, win_size=7)
    norm_img = cv2.normalize(filtered_img, None, 0, 255, cv2.NORM_MINMAX)
    img_uint8 = norm_img.astype(np.uint8)
    img_resized = cv2.resize(img_uint8, target_size, interpolation=cv2.INTER_AREA)
    return img_resized

val_aug = A.Compose([
    A.Resize(height=Config.IMAGE_SIZE[0], width=Config.IMAGE_SIZE[1]),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])

# =====================================================================================
# 2. PyTorch Dataset Class
# =====================================================================================
class OilSpillDataset(Dataset):
    def __init__(self, image_dir, label_dir, preprocess_fn, augmentations=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.preprocess_fn = preprocess_fn
        self.augmentations = augmentations
        self.file_list = sorted([f for f in os.listdir(image_dir) if f in os.listdir(label_dir)])

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        fname = self.file_list[idx]
        img_path = os.path.join(self.image_dir, fname)
        lbl_path = os.path.join(self.label_dir, fname)

        img = self.preprocess_fn(img_path, target_size=Config.IMAGE_SIZE)
        lbl = cv2.imread(lbl_path, cv2.IMREAD_GRAYSCALE)
        if lbl is None:
            raise FileNotFoundError(f"Label not found at {lbl_path}")
        lbl = (lbl > 0).astype(np.uint8)

        img = np.expand_dims(img, axis=-1)

        if self.augmentations:
            augmented = self.augmentations(image=img, mask=lbl)
            img, lbl = augmented['image'], augmented['mask']
        
        return img, lbl.unsqueeze(0).float()

# =====================================================================================
# 3. Model Definition
# =====================================================================================
def build_model():
    config = SegformerConfig.from_pretrained(Config.MODEL_NAME)
    config.num_channels = 1
    config.num_labels = 1
    
    model = SegformerForSemanticSegmentation.from_pretrained(
        Config.MODEL_NAME, config=config, ignore_mismatched_sizes=True
    )

    original_weights = SegformerForSemanticSegmentation.from_pretrained(Config.MODEL_NAME).segformer.encoder.patch_embeddings[0].proj.weight.data
    new_weights = original_weights.mean(dim=1, keepdim=True)
    model.segformer.encoder.patch_embeddings[0].proj.weight.data = new_weights
    
    return model.to(device)

# =====================================================================================
# 4. Evaluation Function
# =====================================================================================
def refine_mask(mask_np):
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, Config.MORPH_KERNEL_SIZE)
    opened_mask = cv2.morphologyEx(mask_np, cv2.MORPH_OPEN, kernel, iterations=1)
    closed_mask = cv2.morphologyEx(opened_mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    return closed_mask

def evaluate_model(model, loader):
    model.eval()
    all_preds, all_masks = [], []
    
    loop = tqdm(loader, desc='Evaluating', leave=False)
    with torch.no_grad():
        for imgs, masks in loop:
            imgs = imgs.to(device)
            
            with autocast(device_type="cuda"):
                outputs = model(pixel_values=imgs).logits
                preds = F.interpolate(outputs, size=masks.shape[2:], mode='bilinear', align_corners=False)

            bin_preds = (torch.sigmoid(preds) > 0.5).cpu().numpy().astype(np.uint8)
            
            if Config.APPLY_POST_PROCESSING:
                refined_preds = np.array([refine_mask(p.squeeze(0)) for p in bin_preds]).astype(np.uint8)
                all_preds.append(refined_preds.flatten())
            else:
                all_preds.append(bin_preds.flatten())
                
            all_masks.append(masks.numpy().flatten())

    preds_arr = np.concatenate(all_preds)
    masks_arr = np.concatenate(all_masks)
    
    # --- Calculate all metrics ---
    acc = accuracy_score(masks_arr, preds_arr)
    p, r, f, _ = precision_recall_fscore_support(masks_arr, preds_arr, average='binary', zero_division=0)
    iou = jaccard_score(masks_arr, preds_arr, average='binary', zero_division=0)
    
    return acc, p, r, f, iou

# =====================================================================================
# 5. Main Execution Block
# =====================================================================================
def main():
    print("--- Evaluating Final Model on Validation Set ---")
    
    # --- 1. Build and Load the final model ---
    model = build_model()
    try:
        state_dict = torch.load(Config.FINAL_MODEL_PATH)
        new_state_dict = OrderedDict()
        for k, v in state_dict.items():
            name = k[7:] if k.startswith('module.') else k
            new_state_dict[name] = v
        model.load_state_dict(new_state_dict)
        print(f"✅ Successfully loaded final model from {Config.FINAL_MODEL_PATH}")
    except FileNotFoundError:
        print(f"Error: Model file not found at {Config.FINAL_MODEL_PATH}. Cannot evaluate.")
        return
    except Exception as e:
        print(f"An error occurred while loading the model: {e}")
        return

    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    # --- 2. Prepare the validation dataloader ---
    val_dataset = OilSpillDataset(
        image_dir=Config.VAL_IMG_DIR,
        label_dir=Config.VAL_LBL_DIR,
        preprocess_fn=enhanced_sar_preprocessing,
        augmentations=val_aug
    )
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    print(f"Loaded {len(val_dataset)} images for final validation.")

    # --- 3. Run evaluation and print metrics ---
    acc, p, r, f, iou = evaluate_model(model, val_loader)

    print(f"\n{'='*20} FINAL MODEL PERFORMANCE {'='*20}")
    print(f"Accuracy:        {acc:.4f}")
    print(f"Precision:       {p:.4f}")
    print(f"Recall:          {r:.4f}")
    print(f"F1-Score:        {f:.4f}")
    print(f"IoU (Jaccard):   {iou:.4f}")
    print(f"{'='*41}")

if __name__ == '__main__':
    main()

Using device: cuda
--- Evaluating Final Model on Validation Set ---


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b4-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- segformer.encoder.patch_embeddings.0.proj.weight: found shape torch.Size([64, 3, 7, 7]) in the checkpoint and torch.Size([64, 1, 7, 7]) in the model instantiated
- decode_head.classifier.weight: found shape torch.Size([150, 768, 1, 1]) in the checkpoint and torch.Size([1, 768, 1, 1]) in the model instantiated
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Successfully loaded final model from /kaggle/working/best_model_continued.pth
Loaded 839 images for final validation.


Evaluating:   0%|          | 0/27 [00:00<?, ?it/s]


==================== FINAL MODEL PERFORMANCE ====================
Accuracy:        0.9405
Precision:       0.8981
Recall:          0.9348
F1-Score:        0.9161
IoU (Jaccard):   0.8451


In [1]:
import pandas as pd
from pathlib import Path

# 1. Define the main path to your dataset
# This path assumes your dataset is in the default Kaggle input directory.
base_path = Path('/kaggle/input/sentinel-dataset/sentinel_dataset/')
train_path = base_path / 'train'
test_path = base_path / 'test'

# 2. Get a list of all file paths in train and test directories
# The rglob('*') function recursively finds all files in all subdirectories.
train_files = list(train_path.rglob('*.*'))
test_files = list(test_path.rglob('*.*'))
all_files = train_files + test_files

# 3. Calculate the total size of the dataset
total_size_bytes = sum(f.stat().st_size for f in all_files)
# Convert bytes to a more readable format (Gigabytes)
total_size_gb = total_size_bytes / (1024 ** 3)

# 4. Print the summary information
print("--- Dataset Overview ---")
print(f"Total number of images: {len(all_files):,}")
print(f"Number of training images: {len(train_files):,}")
print(f"Number of testing images: {len(test_files):,}")
print(f"Total dataset size: {total_size_gb:.2f} GB\n")


# 5. (Optional) Show the distribution of images per class in the training set
print("--- Training Set Class Distribution ---")
class_counts = {}
# Iterate through the subdirectories in the train folder
for class_dir in train_path.iterdir():
    if class_dir.is_dir():
        # Count files in each class subdirectory
        class_counts[class_dir.name] = len(list(class_dir.glob('*.*')))

# Create a Pandas DataFrame for a nice display
class_df = pd.DataFrame(list(class_counts.items()), columns=['Class', 'Image Count'])
print(class_df.to_string(index=False))

--- Dataset Overview ---
Total number of images: 8,386
Number of training images: 6,708
Number of testing images: 1,678
Total dataset size: 0.62 GB

--- Training Set Class Distribution ---
Class  Image Count
label         3354
image         3354
